# DLICV
**Przed uruchomieniem:** Runtime -> Change runtime type -> **T4 GPU**

## 1. Klonowanie repo

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

## 2. Instalacja zaleznosci

In [ ]:
!pip install -q timm==1.0.11 PyYAML==6.0.2 pytest==8.3.3

## 3. Testy

In [ ]:
!pytest tests/test_smoke.py tests/test_transforms.py tests/test_models.py tests/test_metrics.py -q

## 4. Sprawdzenie GPU

In [ ]:
from src.utils.device import get_device, device_info
d = get_device()
print('Device:', device_info(d))
assert d.type == 'cuda', 'GPU nie aktywny! Runtime -> Change runtime type -> T4 GPU.'

## 5. Pobranie Pets

In [ ]:
!python scripts/download_pets.py

## 6. Pierwszy eksperyment: wariant A, ResNet-18, seed 0

**Konfiguracja:** 30 obrazow/klase realne, brak augmentacji, 30 epok cosine, batch=64  

In [ ]:
!python -m src.train --config configs/base.yaml configs/colab.yaml configs/exp/A_resnet18_seed0.yaml

## 7. Podglad wynikow

In [ ]:
import json
from pathlib import Path
import pandas as pd

out = Path('outputs/A_resnet18_seed0')
print('Files:', sorted(p.name for p in out.iterdir()))
print()

df = pd.read_csv(out / 'metrics_metrics.csv')
print('Last 5 epochs:')
print(df.tail())
print()

final = json.loads((out / 'final_results.json').read_text())
print('Final test metrics:')
print(f"  accuracy:         {final['test']['accuracy']:.4f}")
print(f"  balanced accuracy: {final['test']['balanced_accuracy']:.4f}")
print(f"  macro F1:         {final['test']['macro_f1']:.4f}")
print(f"  best val acc:     {final['best_val_acc']:.4f} @ epoch {final['best_epoch']}")

## 8. Wykres krzywej uczenia

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].set_title('Train loss')
axes[1].plot(df['epoch'], df['val_acc'], label='val acc')
axes[1].plot(df['epoch'], df['val_bal_acc'], label='val bal_acc')
axes[1].plot(df['epoch'], df['val_macro_f1'], label='val macro F1')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('metric'); axes[1].legend(); axes[1].set_title('Val metrics')
plt.tight_layout(); plt.show()